In [1]:
!pip install -q transformers torch gradio accelerate sentencepiece

In [2]:
import torch
import gradio as gr
import gc
import warnings

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline
)

warnings.filterwarnings("ignore")

print("="*50)
print("Universal NLP Assistant - Setup Complete")
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("="*50)

Universal NLP Assistant - Setup Complete
PyTorch: 2.10.0+cpu
CUDA Available: False


In [3]:
class UniversalNLPAssistant:

    def __init__(self):

        print("Loading models...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        device_num = 0 if self.device == "cuda" else -1

        # BERT pipelines
        print("Loading BERT pipelines...")
        self.sentiment_pipeline = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=device_num
        )

        self.ner_pipeline = pipeline(
            "ner",
            model="dslim/bert-base-NER",
            aggregation_strategy="simple",
            device=device_num
        )

        # GPT2
        print("Loading GPT2...")
        self.gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
        self.gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2").to(self.device)

        self.gpt2_tokenizer.pad_token = self.gpt2_tokenizer.eos_token

        # T5
        print("Loading T5...")
        self.t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
        self.t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(self.device)

        print("✅ Models Loaded Successfully")


assistant = UniversalNLPAssistant()

Loading models...
Loading BERT pipelines...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Models Loaded Successfully


In [4]:
class ModelFunctions:

    # BERT Sentiment
    def sentiment(text):
        return assistant.sentiment_pipeline(text)

    # BERT NER
    def ner(text):
        return assistant.ner_pipeline(text)

    # GPT2 Story
    def generate(prompt):

        inputs = assistant.gpt2_tokenizer.encode(
            prompt,
            return_tensors="pt"
        ).to(assistant.device)

        output = assistant.gpt2_model.generate(
            inputs,
            max_length=80,
            temperature=0.8,
            do_sample=True
        )

        text = assistant.gpt2_tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )

        return text

    # T5 Summarize
    def summarize(text):

        input_text = "summarize: " + text

        inputs = assistant.t5_tokenizer.encode(
            input_text,
            return_tensors="pt",
            truncation=True
        ).to(assistant.device)

        outputs = assistant.t5_model.generate(
            inputs,
            max_length=60
        )

        summary = assistant.t5_tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        return summary

    # T5 Translate
    def translate(text):

        input_text = "translate English to French: " + text

        inputs = assistant.t5_tokenizer.encode(
            input_text,
            return_tensors="pt"
        ).to(assistant.device)

        outputs = assistant.t5_model.generate(inputs)

        return assistant.t5_tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )


In [5]:
with gr.Blocks(title="Universal NLP Assistant") as demo:

    gr.Markdown("# 🚀 Universal NLP Assistant")
    gr.Markdown("BERT + GPT2 + T5 Demo")

    with gr.Tab("Sentiment (BERT)"):
        text = gr.Textbox()
        out = gr.JSON()
        btn = gr.Button("Analyze")

        btn.click(ModelFunctions.sentiment, text, out)

    with gr.Tab("NER (BERT)"):
        text2 = gr.Textbox()
        out2 = gr.JSON()
        btn2 = gr.Button("Extract")

        btn2.click(ModelFunctions.ner, text2, out2)

    with gr.Tab("Story Generation (GPT2)"):
        prompt = gr.Textbox()
        story = gr.Textbox()
        btn3 = gr.Button("Generate")

        btn3.click(ModelFunctions.generate, prompt, story)

    with gr.Tab("Summarization (T5)"):
        text3 = gr.Textbox(lines=5)
        summary = gr.Textbox()
        btn4 = gr.Button("Summarize")

        btn4.click(ModelFunctions.summarize, text3, summary)

    with gr.Tab("Translation (T5)"):
        text4 = gr.Textbox()
        trans = gr.Textbox()
        btn5 = gr.Button("Translate")

        btn5.click(ModelFunctions.translate, text4, trans)

In [6]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a640b9bf5175a8cb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
